* Lấy data: 


In [1]:
import pandas as pd

campaign = pd.read_csv('../data_clean/campaign_clean_info.csv')
customer = pd.read_csv('../data_clean/customer_response_clean.csv')
sales = pd.read_csv('../data_clean/sales_conversion_clean.csv')

Tính tỷ lệ chuyển đổi theo từng kênh Marketing

In [2]:
df_campaign_sales = pd.merge(campaign,sales, on='campaign_id', how='inner')

converted_channel = (
    df_campaign_sales
    .groupby('channel')['converted']
    .mean()
    .reset_index(name='converted_rate')

)
converted_channel



,channel,converted_rate
0,Email,0.428571
1,Facebook,0.682927
2,Google,0.708333
3,TikTok,0.428571


- Tính doanh thu trung bình theo chiến dịch: 

In [3]:
mean_revenue_sales_to_campaign = (
    df_campaign_sales
    .groupby('campaign_id')['amount']
    .mean()
    .reset_index(name='revenue_sales')

)

mean_revenue_sales_to_campaign

,campaign_id,revenue_sales
0,CMP100,237500.000000
1,CMP101,100000.000000
2,CMP102,238888.888889
3,CMP103,128571.428571
4,CMP104,268750.000000
5,CMP105,266666.666667
6,CMP106,137500.000000
7,CMP107,250000.000000
8,CMP108,285000.000000
9,CMP109,292857.142857


- Chiến dịch hiệu quả nhất: 

In [4]:
campaign_hieu_qua = mean_revenue_sales_to_campaign.sort_values(by='revenue_sales', ascending=False).head()
campaign_hieu_qua

,campaign_id,revenue_sales
19,CMP119,462500.000000
10,CMP110,350000.000000
15,CMP115,333333.333333
14,CMP114,310000.000000
17,CMP117,300000.000000


- Tổng hợp số lương phản hồi: 

In [5]:
tong_so_luong_phan_hoi = (
    customer
    .groupby('response')
    .size()
    .reset_index(name='quantity_response')
)
tong_so_luong_phan_hoi.head(20)

,response,quantity_response
0,Click,36
1,Ignore,48
2,View,36


- Tìm nhóm chiến dịch có ngân sách cao nhất nhưng hiệu quả thấp nhất: 
 

In [6]:
conversion_by_campaign = (
    df_campaign_sales
    .groupby('campaign_id')['converted']
    .mean()
    .reset_index(name='conversion_rate')
)

conversion_by_campaign

avg_budget = campaign['budget'].mean()
avg_conversion_rate = conversion_by_campaign['conversion_rate'].mean()

campaign_analysis = pd.merge(
    campaign,
    conversion_by_campaign,
    on='campaign_id',
    how='left'
)

campaign_analysis


ineffective_campaigns = campaign_analysis[
    (campaign_analysis['budget'] > avg_budget) &
    (campaign_analysis['conversion_rate'] < avg_conversion_rate)
]

ineffective_campaigns


,Unnamed: 0,campaign_id,channel,budget,conversion_rate
3,3,CMP103,Facebook,10000000,0.375000
8,8,CMP108,Email,10000000,0.333333
10,10,CMP110,Google,8000000,0.000000
13,13,CMP113,TikTok,8000000,0.000000
